In [1]:
!uv tree

Resolved 178 packages in 2ms
langchain-project v0.1.0
├── httpx v0.28.1
│   ├── anyio v4.12.0
│   │   └── idna v3.11
│   ├── certifi v2025.11.12
│   ├── httpcore v1.0.9
│   │   ├── certifi v2025.11.12
│   │   └── h11 v0.16.0
│   └── idna v3.11
├── ipywidgets v8.1.8
│   ├── comm v0.2.3
│   ├── ipython v9.8.0
│   │   ├── decorator v5.2.1
│   │   ├── ipython-pygments-lexers v1.1.1
│   │   │   └── pygments v2.19.2
│   │   ├── jedi v0.19.2
│   │   │   └── parso v0.8.5
│   │   ├── matplotlib-inline v0.2.1
│   │   │   └── traitlets v5.14.3
│   │   ├── pexpect v4.9.0
│   │   │   └── ptyprocess v0.7.0
│   │   ├── prompt-toolkit v3.0.52
│   │   │   └── wcwidth v0.2.14
│   │   ├── pygments v2.19.2
│   │   ├── stack-data v0.6.3
│   │   │   ├── asttokens v3.0.1
│   │   │   ├── executing v2.2.1
│   │   │   └── pure-eval v0.2.3
│   │   └── traitlets v5.14.3
│   ├── jupyterlab-widgets v3.0.16
│   ├── traitlets v5.14.3
│   └── widgetsnbextension v4.0.15
├── langchain v1.2.0
│   ├── langchain-core v1.2.

In [2]:
from load_dotenv import load_dotenv
load_dotenv()

True

In [3]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "../example_data/nke-10k-2023.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

In [4]:
len(docs), type(docs[0]), docs[0].metadata

(107,
 langchain_core.documents.base.Document,
 {'producer': 'EDGRpdf Service w/ EO.Pdf 22.0.40.0',
  'creator': 'EDGAR Filing HTML Converter',
  'creationdate': '2023-07-20T16:22:00-04:00',
  'title': '0000320187-23-000039',
  'author': 'EDGAR Online, a division of Donnelley Financial Solutions',
  'subject': 'Form 10-K filed on 2023-07-20 for the period ending 2023-05-31',
  'keywords': '0000320187-23-000039; ; 10-K',
  'moddate': '2023-07-20T16:22:08-04:00',
  'source': '../example_data/nke-10k-2023.pdf',
  'total_pages': 107,
  'page': 0,
  'page_label': '1'})

In [5]:
len(docs[0].page_content), docs[0].page_content[:300]

(3645,
 'Table of Contents\nUNITED STATES\nSECURITIES AND EXCHANGE COMMISSION\nWashington, D.C. 20549\nFORM 10-K\n(Mark One)\n☑  ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(D) OF THE SECURITIES EXCHANGE ACT OF 1934\nFOR THE FISCAL YEAR ENDED MAY 31, 2023\nOR\n☐  TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(D) OF THE')

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(docs)

len(docs), len(all_splits)

(107, 516)

In [7]:
from langchain_openai import OpenAIEmbeddings  # or AzureOpenAIEmbeddings
import os

embeddings = OpenAIEmbeddings(
    model="qwen/qwen3-embedding-8b",
    api_key=os.getenv("OPENROUTER_API_KEY", ""),
    base_url="https://openrouter.ai/api/v1",
)
vector_1 = embeddings.embed_query(all_splits[0].page_content)
vector_2 = embeddings.embed_query(all_splits[1].page_content)

In [8]:
assert len(vector_1) == len(vector_2)
print(f"Generated vectors of length {len(vector_1)}\n")
print(vector_1[:10])

Generated vectors of length 4096

[0.012054750695824623, 0.008690634742379189, -0.005186346359550953, -0.016470154747366905, 0.022988129407167435, -0.011283807456493378, -0.0007490415591746569, -0.0010162435937672853, 0.007954733446240425, 0.021306071430444717]


In [9]:
from langchain_milvus import Milvus
URI = "./milvus_example.db"
vector_store = Milvus(
    embedding_function=embeddings,
    connection_args={"uri": URI},
    index_params={"index_type": "FLAT", "metric_type": "L2"},
)

/Users/miniyk/Documents/code/python_project/pytest_lesson/langchain_project/.venv/lib/python3.13/site-packages/milvus_lite/__init__.py:15: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [10]:
ids = vector_store.add_documents(documents=all_splits)

/Users/miniyk/Documents/code/python_project/pytest_lesson/langchain_project/.venv/lib/python3.13/site-packages/langchain_milvus/vectorstores/milvus.py:1340: UserWarning: No ids provided and auto_id is False. Setting auto_id to True automatically.
  warnings.warn(


In [11]:
len(all_splits)

516

In [ ]:
results = vector_store.similarity_search(
    "How many distribution centers does Nike have in the US?"
)

In [13]:
len(results), results[0]

(4,
 Document(metadata={'author': 'EDGAR Online, a division of Donnelley Financial Solutions', 'creationdate': '2023-07-20T16:22:00-04:00', 'creator': 'EDGAR Filing HTML Converter', 'keywords': '0000320187-23-000039; ; 10-K', 'moddate': '2023-07-20T16:22:08-04:00', 'page': 3, 'page_label': '4', 'pk': 463324034759917580, 'producer': 'EDGRpdf Service w/ EO.Pdf 22.0.40.0', 'source': '../example_data/nke-10k-2023.pdf', 'start_index': 2264, 'subject': 'Form 10-K filed on 2023-07-20 for the period ending 2023-05-31', 'title': '0000320187-23-000039', 'total_pages': 107}, page_content='We also sell sports apparel, which features the same trademarks and are sold predominantly through the same marketing and distribution channels as athletic footwear.\nOur sports apparel, similar to our athletic footwear products, is designed primarily for athletic use, although many of the products are worn for casual or leisure purposes,\nand demonstrates our commitment to innovation and high-quality constructi

In [14]:
results = await vector_store.asimilarity_search("When was Nike incorporated?")


In [17]:
results = vector_store.similarity_search_with_score("What was Nike's revenue in 2023?")
doc, score = results[0]
print(f"Score: {score}\n")
print(f"Score: {results[1][1]}\n")

Score: 1.0252101421356201

Score: 1.0372899770736694



In [18]:
embedding = embeddings.embed_query("How were Nike's margins impacted in 2023?")

results = vector_store.similarity_search_by_vector(embedding)
print(results[0])

page_content='EMPLOYEE STOCK PURCHASE PLANS
In addition to the Stock Incentive Plan, the Company gives employees the right to purchase shares at a discount from the market price under ESPPs. Subject to the
annual statutory limit, employees are eligible to participate through payroll deductions of up to 10% of their compensation. At the end of each six-month offering period,
shares are purchased by the participants at 85% of the lower of the fair market value at the beginning or the end of the offering period. Employees purchased 3.0 million,
2.0 million and 2.5 million shares during each of the fiscal years ended May 31, 2023, 2022 and 2021, respectively.
RESTRICTED STOCK AND RESTRICTED STOCK UNITS
Recipients of restricted stock are entitled to cash dividends and to vote their respective shares throughout the period of restriction. Recipients of restricted stock units,' metadata={'author': 'EDGAR Online, a division of Donnelley Financial Solutions', 'creationdate': '2023-07-20T16:22:00

In [ ]:
from typing import List

from langchain_core.documents import Document
from langchain_core.runnables import chain

In [32]:
from typing import List

from langchain_core.documents import Document
from langchain_core.runnables import chain

@chain
def retriever(query: str) -> List[Document]:
    return vector_store.similarity_search(query, k=1)


results = retriever.batch(
    [
        "How many distribution centers does Nike have in the US?",
        "When was Nike incorporated?",
    ],
)

In [34]:
type(results), len(results), len(results[0])

(list, 2, 1)

In [35]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 1},
)

retriever.batch(
    [
        "How many distribution centers does Nike have in the US?",
        "When was Nike incorporated?",
    ],
)

[[Document(metadata={'author': 'EDGAR Online, a division of Donnelley Financial Solutions', 'creationdate': '2023-07-20T16:22:00-04:00', 'creator': 'EDGAR Filing HTML Converter', 'keywords': '0000320187-23-000039; ; 10-K', 'moddate': '2023-07-20T16:22:08-04:00', 'page': 3, 'page_label': '4', 'pk': 463324034759917580, 'producer': 'EDGRpdf Service w/ EO.Pdf 22.0.40.0', 'source': '../example_data/nke-10k-2023.pdf', 'start_index': 2264, 'subject': 'Form 10-K filed on 2023-07-20 for the period ending 2023-05-31', 'title': '0000320187-23-000039', 'total_pages': 107}, page_content='We also sell sports apparel, which features the same trademarks and are sold predominantly through the same marketing and distribution channels as athletic footwear.\nOur sports apparel, similar to our athletic footwear products, is designed primarily for athletic use, although many of the products are worn for casual or leisure purposes,\nand demonstrates our commitment to innovation and high-quality construction.

In [1]:
def Singleton(cls):
    instance = {}

    def _singleton_wrapper(*args, **kargs):
        if cls not in instance:
            instance[cls] = cls(*args, **kargs)
        return instance[cls]

    return _singleton_wrapper


@Singleton
class SingletonTest(object):
    def __init__(self, name):
        self.name = name


slt_1 = SingletonTest('第1次创建')
print(slt_1.name)
slt_2 = SingletonTest('第2次创建')
print(slt_1.name, slt_2.name)

print(slt_1 is slt_2)



第1次创建
第1次创建 第1次创建
True
